# Page Keyword Expansion

Use this notebook to compare a high-value Lifeline page's current GSC query footprint with adjacent keyword ideas from DataForSEO.

### Quick start
1. Run the **Setup (run once)** cell.
2. In **Parameters**, paste a full Lifeline page URL.
3. Keep `RUN_MODE="sandbox"` while testing.
4. Review the spend preview before enabling live mode.
5. Run the remaining cells from top to bottom.


In [1]:
#@title Setup (run once)
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidanm-lla/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q "plotly>=6.1.1" "kaleido>=1.2.0"
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
from google.cloud import bigquery

from lla_data import config
from lla_data.bq import build_date_params, default_query_window, get_client, load_sql_template, run_query
from lla_data.dataforseo import (
    canonicalize_lifeline_url,
    extract_lifeline_page_path,
    get_user_data,
    keywords_for_site,
    response_to_keyword_frame,
)

client = get_client()


In [ ]:
#@title Parameters
PAGE_URL = "https://www.lifeline.org.au/get-help/support-toolkit/anxiety/" #@param {type:"string"}
DAYS_BACK = 92 #@param {type:"integer"}
LOCATION_NAME = config.DEFAULT_KEYWORD_LOCATION #@param {type:"string"}
LANGUAGE_NAME = config.DEFAULT_KEYWORD_LANGUAGE #@param {type:"string"}
RUN_MODE = "live" #@param ["sandbox", "live"]
RUN_LIVE = True #@param {type:"boolean"}
USE_CACHE = True #@param {type:"boolean"}
MIN_SEARCH_VOLUME = 50 #@param {type:"integer"}
MAX_RESULTS = 100 #@param {type:"integer"}

window = default_query_window(DAYS_BACK)
page_url = canonicalize_lifeline_url(PAGE_URL)
page_path = extract_lifeline_page_path(PAGE_URL)

print(f"Page URL: {page_url}")
print(f"Page path: {page_path}")
print(f"GSC window: {window.start_date} to {window.end_date}")
print(f"DataForSEO market: {LOCATION_NAME} / {LANGUAGE_NAME}")


Page URL: https://www.lifeline.org.au/get-help/support-toolkit/anxiety
Page path: /get-help/support-toolkit/anxiety
GSC window: 2025-12-10 to 2026-03-12
DataForSEO market: Australia / English


In [3]:
# Guardrails / spend preview
print("Expected DataForSEO calls for this run: 1 keywords_for_site request")
print(f"Run mode: {RUN_MODE}")
print(f"Use cache: {USE_CACHE}")

if RUN_MODE == "live" and not RUN_LIVE:
    raise ValueError("Live DataForSEO calls are blocked. Review the request first, then set RUN_LIVE=True.")

try:
    account = get_user_data(mode="sandbox", use_cache=True)
    account_result = account.payload["tasks"][0]["result"][0]
    print(f"DataForSEO account: {account_result.get('login')}")
    print(f"User-data cache: {account.path}")
except Exception as exc:
    print(f"Could not load DataForSEO account info: {exc}")


Expected DataForSEO calls for this run: 1 keywords_for_site request
Run mode: live
Use cache: True


ValueError: Live DataForSEO calls are blocked. Review the request first, then set RUN_LIVE=True.

In [ ]:
# Current query drivers from GSC
query_sql = load_sql_template(
    "sql/notebooks/seo_query_drivers_by_page.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
)

query_params = build_date_params(window) + [
    bigquery.ScalarQueryParameter("page_path", "STRING", page_path),
]

df_queries = run_query(client, query_sql, params=query_params)
current_query_drivers = (
    df_queries.groupby("query", as_index=False)
    .agg(
        clicks=("clicks", "sum"),
        impressions=("impressions", "sum"),
        avg_position=("avg_position", "mean"),
    )
    .sort_values(["clicks", "impressions"], ascending=[False, False])
)
current_query_drivers["ctr"] = (
    current_query_drivers["clicks"]
    / current_query_drivers["impressions"].replace(0, pd.NA)
)
current_query_drivers.head(20)


In [ ]:
# External keyword ideas from DataForSEO
keyword_response = keywords_for_site(
    page_url,
    location_name=LOCATION_NAME,
    language_name=LANGUAGE_NAME,
    mode=RUN_MODE,
    use_cache=USE_CACHE,
    run_live=RUN_LIVE,
)
print(f"Keyword response cache: {keyword_response.path}")
print(f"Loaded from cache: {keyword_response.from_cache}")

keyword_opportunities = response_to_keyword_frame(keyword_response)
keyword_opportunities = keyword_opportunities.dropna(subset=["keyword"]).copy()
keyword_opportunities = keyword_opportunities[
    keyword_opportunities["search_volume"].fillna(0) >= MIN_SEARCH_VOLUME
]
keyword_opportunities = keyword_opportunities.sort_values("search_volume", ascending=False)
keyword_opportunities = keyword_opportunities.head(MAX_RESULTS).reset_index(drop=True)
keyword_opportunities.head(20)


In [ ]:
# Compare current visibility vs adjacent opportunities
gsc_lookup = (
    current_query_drivers.assign(query_norm=lambda df: df["query"].str.strip().str.lower())
    .set_index("query_norm")[["clicks", "impressions", "ctr", "avg_position"]]
)

related_keyword_opportunities = keyword_opportunities.copy()
related_keyword_opportunities["keyword_norm"] = related_keyword_opportunities["keyword"].str.strip().str.lower()
related_keyword_opportunities = related_keyword_opportunities.join(
    gsc_lookup,
    on="keyword_norm",
    rsuffix="_gsc",
)
related_keyword_opportunities["already_visible_in_gsc"] = related_keyword_opportunities["clicks"].fillna(0) > 0
related_keyword_opportunities["high_volume_low_current_visibility"] = (
    (related_keyword_opportunities["search_volume"].fillna(0) >= 500)
    & (related_keyword_opportunities["clicks"].fillna(0) < 5)
)
related_keyword_opportunities["new_opportunity"] = ~related_keyword_opportunities["already_visible_in_gsc"]
related_keyword_opportunities["intent_review"] = "Review SERP intent before expanding page"

display_columns = [
    "keyword",
    "search_volume",
    "competition",
    "competition_index",
    "cpc",
    "clicks",
    "impressions",
    "avg_position",
    "already_visible_in_gsc",
    "new_opportunity",
    "high_volume_low_current_visibility",
    "intent_review",
    "monthly_searches",
]

related_keyword_opportunities[display_columns].head(30)


In [ ]:
# Top opportunities not already visible in GSC
top_new_opportunities = related_keyword_opportunities[
    related_keyword_opportunities["new_opportunity"]
][display_columns].sort_values(
    ["high_volume_low_current_visibility", "search_volume"],
    ascending=[False, False],
)

top_new_opportunities.head(30)


## How to interpret the tables

- Expand the current page when the keyword intent is close and the page already partially matches the need.
- Add a supporting section when the opportunity is adjacent but narrower than the current page's main purpose.
- Do not target the term from this page when the SERP intent is materially different or the page would become unfocused.
- Use `monthly_searches` to sense seasonality before deciding whether the opportunity is evergreen or campaign-like.
